In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)

    for node in bn.nodes:
        cpd = bn.cpds[node]
        
        probs = bn.predict_probability(data, node)
        
        true_values = data[node]
        
        y_dummies = pd.get_dummies(true_values)
        
        y_dummies = y_dummies.reindex(columns=probs.columns, fill_value=0)
        
        p_actual = (probs * y_dummies).sum(axis=1)
        
        log_likelihood += np.sum(np.log(p_actual + 1e-9))

        num_parent_configs = cpd.shape[0]
        num_node_states = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)

    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture # <--- NEW IMPORT
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [4]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn_no_context.csv")

df_de = df_de.drop(["Unnamed: 0"], axis = 1)

df_de

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,30902.298694,26.46,77.264286,6.710000,6.550000,11.82,10.039,6.389770,17.555365,-0.168868,1.46,-1.2,15.2,0.000010,581.657956,6.196333
1,31539.117534,26.06,77.264286,6.670000,6.540000,10.93,10.072,5.783688,17.951098,-0.295519,0.81,-2.6,15.2,0.000010,599.683352,10.012677
2,30461.535107,23.87,73.200000,6.610000,6.490000,11.31,10.851,4.768184,19.597935,-2.070729,1.63,-2.4,15.5,0.000009,584.358974,37.999363
3,31990.195768,26.06,74.000000,6.610000,6.420000,10.42,11.667,5.103897,19.176892,-3.199506,2.48,-0.6,15.6,0.000010,575.287356,34.793707
4,34229.966763,26.31,75.400000,6.450000,6.150000,8.93,12.470,4.852664,18.693432,-0.370659,2.55,1.3,15.8,0.000010,585.349446,39.857009
5,34819.413660,26.58,75.800000,6.350000,6.090000,8.12,13.549,6.037145,18.897792,0.092987,1.82,2.4,16.1,0.000010,579.634465,40.920931
6,35554.076381,25.91,76.300000,6.080000,6.010000,7.82,13.757,5.772928,19.247998,0.204270,1.76,3.0,16.1,0.000010,587.141971,52.698567
7,36861.914611,26.01,76.700000,5.940000,5.880000,7.61,14.381,6.537450,19.210129,0.684115,0.44,5.3,16.7,0.000010,594.640652,39.295143
8,37774.244301,25.92,77.000000,5.950000,5.720000,7.33,14.901,7.417530,19.312236,0.679371,1.82,12.0,16.7,0.000010,580.046517,27.981817
9,38811.920567,26.22,77.600000,5.900000,5.600000,7.08,14.885,7.263048,19.519072,0.736708,1.64,4.2,16.5,0.000010,588.706740,8.995473


In [5]:
de = pd.DataFrame()
de = df_de.copy()
de['PC_1'] = de['PC'].shift(1)
de['GGFC_2'] = de['GGFC'].shift(2)
de['FDI_2'] = de['FDI'].shift(2)
de['ARP_1'] = de['ARP'].shift(1)
de['PAM_3'] = de['PAM'].shift(3)
de['EBSG_3'] = de['EBSG'].shift(3)
de['CO2C_2'] = de['CO2C'].shift(2)

In [6]:
de = de.dropna()

In [7]:
discretised_de = pd.DataFrame(index=de.index)

for col in de.columns:
    no_unique = de[col].nunique()
    
    if no_unique <= 1:
        discretised_de[col] = 0
    else:
        try:
            discretised_de[col] = pd.qcut(
                de[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_de[col] = de[col].rank(method='dense').astype(int) - 1
    num_states = discretised_de[col].nunique()
    
    if num_states == 2:
        discretised_de[col] = discretised_de[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_de[col] = discretised_de[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_de[col] = discretised_de[col].astype(str)

print("Discretised data:")
print(discretised_de)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD  ...  CO2C  \
3    Low  High   Low  High  High  High   Low   Low   Low   Low  ...  High   
4    Low  High   Low  High  High  High   Low   Low   Low   Low  ...  High   
5    Low  High   Low  High  High  High   Low  High   Low   Low  ...  High   
6    Low   Low   Low   Low  High  High   Low   Low   Low   Low  ...  High   
7    Low  High   Low   Low   Low  High   Low  High   Low  High  ...  High   
8    Low   Low   Low   Low   Low   Low   Low  High   Low  High  ...  High   
9    Low  High   Low   Low   Low   Low   Low  High  High  High  ...   Low   
10  High  High  High   Low   Low   Low  High  High   Low  High  ...   Low   
11  High   Low  High   Low   Low   Low  High  High  High  High  ...   Low   
12  High   Low  High   Low   Low   Low  High   Low  High  High  ...   Low   
13  High   Low  High  High   Low  High  High   Low  High   Low  ...   Low   
14  High   Low  High  High   Low   Low  High   Low  High  

In [9]:
sm = StructureModel()

In [10]:
sm.add_edges_from([  
 ('EUPC', 'VABI'),
    ('GGFC', 'VABI'),
    ('YUA', 'QOA'),
    ('YUA', 'QOR'),
    ('REST', 'YUA'),
    ('GGFC', 'REST'),
    ('CO2C_2', 'GGFC'),
    ('GGFC', 'LBGD'),
    ('EUPC', 'EBSG'),
    ('EUPC', 'PC'),
    ('QOA', 'EUPC'),
    ('PC_1', 'EUPC'),
    ('GGFC_2', 'EUPC'),
    ('FDI_2', 'EUPC'),
    ('ARP_1', 'PC'),
])



In [11]:
sm.edges

OutEdgeView([('EUPC', 'VABI'), ('EUPC', 'EBSG'), ('EUPC', 'PC'), ('GGFC', 'VABI'), ('GGFC', 'REST'), ('GGFC', 'LBGD'), ('YUA', 'QOA'), ('YUA', 'QOR'), ('QOA', 'EUPC'), ('REST', 'YUA'), ('CO2C_2', 'GGFC'), ('PC_1', 'EUPC'), ('GGFC_2', 'EUPC'), ('FDI_2', 'EUPC'), ('ARP_1', 'PC')])

In [12]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [15]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_de)

bn.fit_cpds(discretised_de, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_de)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 8225.71 (LogLikelihood: -4041.04)


In [17]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'VABI'],
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'VABI']
  - Removing direct edge: EUPC -> VABI
LV Model BIC: 8846.59 (LogLikelihood: -4310.44)


In [20]:
bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'PC'], 
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'PC']
  - Removing direct edge: EUPC -> PC
LV Model BIC: 8846.59 (LogLikelihood: -4310.44)


In [22]:
bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['QOA', 'EUPC'],  # Replace with your specific nodes
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['QOA', 'EUPC']
  - Removing direct edge: QOA -> EUPC
LV Model BIC: 8774.78 (LogLikelihood: -4310.44)


In [23]:
bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['FDI_2', 'EUPC'],  # Replace with your specific nodes
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['FDI_2', 'EUPC']
  - Removing direct edge: FDI_2 -> EUPC
LV Model BIC: 8769.65 (LogLikelihood: -4310.44)


In [25]:
bn_with_lv_5, data_with_lv_5, bic_lv_5 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'EBSG'],  # Replace with your specific nodes
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'EBSG']
  - Removing direct edge: EUPC -> EBSG
LV Model BIC: 8846.59 (LogLikelihood: -4310.44)


In [27]:
bn_with_lv_6, data_with_lv_6, bic_lv_6 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['PC_1', 'EUPC'],  # Replace with your specific nodes
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['PC_1', 'EUPC']
  - Removing direct edge: PC_1 -> EUPC
LV Model BIC: 8769.65 (LogLikelihood: -4310.44)


In [28]:
bn_with_lv_7, data_with_lv_7, bic_lv_7 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['GGFC_2', 'EUPC'],  # Replace with your specific nodes
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['GGFC_2', 'EUPC']
  - Removing direct edge: GGFC_2 -> EUPC
LV Model BIC: 8769.65 (LogLikelihood: -4310.44)


In [29]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [30]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn_no_context.csv")

df_de = df_de.drop(["Unnamed: 0"], axis = 1)

df_de

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,30902.298694,26.46,77.264286,6.710000,6.550000,11.82,10.039,6.389770,17.555365,-0.168868,1.46,-1.2,15.2,0.000010,581.657956,6.196333
1,31539.117534,26.06,77.264286,6.670000,6.540000,10.93,10.072,5.783688,17.951098,-0.295519,0.81,-2.6,15.2,0.000010,599.683352,10.012677
2,30461.535107,23.87,73.200000,6.610000,6.490000,11.31,10.851,4.768184,19.597935,-2.070729,1.63,-2.4,15.5,0.000009,584.358974,37.999363
3,31990.195768,26.06,74.000000,6.610000,6.420000,10.42,11.667,5.103897,19.176892,-3.199506,2.48,-0.6,15.6,0.000010,575.287356,34.793707
4,34229.966763,26.31,75.400000,6.450000,6.150000,8.93,12.470,4.852664,18.693432,-0.370659,2.55,1.3,15.8,0.000010,585.349446,39.857009
5,34819.413660,26.58,75.800000,6.350000,6.090000,8.12,13.549,6.037145,18.897792,0.092987,1.82,2.4,16.1,0.000010,579.634465,40.920931
6,35554.076381,25.91,76.300000,6.080000,6.010000,7.82,13.757,5.772928,19.247998,0.204270,1.76,3.0,16.1,0.000010,587.141971,52.698567
7,36861.914611,26.01,76.700000,5.940000,5.880000,7.61,14.381,6.537450,19.210129,0.684115,0.44,5.3,16.7,0.000010,594.640652,39.295143
8,37774.244301,25.92,77.000000,5.950000,5.720000,7.33,14.901,7.417530,19.312236,0.679371,1.82,12.0,16.7,0.000010,580.046517,27.981817
9,38811.920567,26.22,77.600000,5.900000,5.600000,7.08,14.885,7.263048,19.519072,0.736708,1.64,4.2,16.5,0.000010,588.706740,8.995473


In [31]:
de = pd.DataFrame()
de = df_de.copy()
de['EUPC_3'] = de['EUPC'].shift(3)
de['PAM_3'] = de['PAM'].shift(3)
de['EBSG_3'] = de['EBSG'].shift(3)

In [32]:
de = de.dropna()

In [33]:
discretised_de = pd.DataFrame(index=de.index)

for col in de.columns:
    no_unique = de[col].nunique()
    
    if no_unique <= 1:
        discretised_de[col] = 0
    else:
        try:
            discretised_de[col] = pd.qcut(
                de[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_de[col] = de[col].rank(method='dense').astype(int) - 1
    num_states = discretised_de[col].nunique()
    
    if num_states == 2:
        discretised_de[col] = discretised_de[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_de[col] = discretised_de[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_de[col] = discretised_de[col].astype(str)

print("Discretised data:")
print(discretised_de)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
3    Low  High   Low  High  High  High   Low   Low   Low   Low  High   Low   
4    Low  High   Low  High  High  High   Low   Low   Low   Low  High   Low   
5    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
6    Low   Low   Low   Low  High  High   Low   Low   Low   Low   Low  High   
7    Low  High   Low   Low   Low  High   Low  High   Low  High   Low  High   
8    Low   Low   Low   Low   Low   Low   Low  High   Low  High   Low  High   
9    Low  High   Low   Low   Low   Low   Low  High  High  High   Low  High   
10  High  High  High   Low   Low   Low  High  High   Low  High  High  High   
11  High   Low  High   Low   Low   Low  High  High  High  High  High  High   
12  High   Low  High   Low   Low   Low  High   Low  High  High   Low   Low   
13  High   Low  High  High   Low  High  High   Low  High   Low  High   Low   
14  High   Low  High  High   Low   Low  High  

In [34]:
sm = StructureModel()

In [35]:
sm.add_edges_from([
     ('EUPC_3', 'ARP'),
    ('PAM_3', 'ARP'),
    ('FDI', 'ARP'),
    ('EBSG_3', 'FDI')
])

In [36]:
sm.edges

OutEdgeView([('EUPC_3', 'ARP'), ('PAM_3', 'ARP'), ('FDI', 'ARP'), ('EBSG_3', 'FDI')])

In [37]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [38]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_de)

bn.fit_cpds(discretised_de, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_de)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 2735.06 (LogLikelihood: -1347.01)


In [39]:
bn_with_lv, data_with_lv, bic_lv = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_3', 'ARP'],  # Replace with your specific nodes
    discretised_data=discretised_de,
    raw_data=de
)

Estimating LV based on continuous GMM distributions in: ['EUPC_3', 'ARP']
  - Removing direct edge: EUPC_3 -> ARP
LV Model BIC: 3279.00 (LogLikelihood: -1616.41)
